# 📊 Notebook 01 — Data Exploration and Preprocessing

This notebook downloads, cleans, explores, and freezes both Amharic datasets:
1. **Sentiment** (AfriSenti, ~9,480 tweets, 3-class)
2. **Summarization** (XL-Sum Amharic, 500-pair evaluation set)

All preprocessing is deterministic (seed = 42) and produces the same output files
as `python scripts/prepare_data.py`.

In [ ]:
# ── Install dependencies (Colab) ──
import importlib, subprocess, sys
for pkg in ['datasets', 'transformers', 'peft', 'scikit-learn', 'sacrebleu',
            'bert-score', 'codecarbon', 'pyyaml', 'sentencepiece', 'accelerate']:
    if not importlib.util.find_spec(pkg.replace('-', '_')):
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

In [ ]:
import os, sys, random, hashlib, unicodedata, re, json
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

# If running from notebooks/, add the repo root to sys.path
REPO = Path('.').resolve()
if REPO.name == 'notebooks':
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
os.chdir(REPO)

from src.utils import load_config, set_all_seeds
cfg = load_config()
SEED = cfg['experiment']['seeds'][0]
set_all_seeds(SEED)
print(f'Repo root: {REPO}\nSeed: {SEED}')

## 1. Load Raw Sentiment Data

In [ ]:
from datasets import load_dataset

scfg = cfg['datasets']['sentiment']
ds_sent = None
for cand in scfg['hf_candidates']:
    try:
        ds_sent = load_dataset(cand['path'], cand.get('name'), trust_remote_code=True)
        print(f'Loaded: {cand}')
        break
    except Exception as e:
        print(f'  skip {cand}: {e}')
print(ds_sent)

In [ ]:
# Pool all splits into one DataFrame
frames = []
text_col = scfg['text_column'] if scfg['text_column'] in ds_sent['train'].column_names else 'text'
for split_name, split in ds_sent.items():
    feat = split.features[scfg['label_column']]
    labels = [feat.names[v].lower() for v in split[scfg['label_column']]] if hasattr(feat, 'names') \
             else [str(v).lower().strip() for v in split[scfg['label_column']]]
    frames.append(pd.DataFrame({'text': split[text_col], 'label': labels, 'orig_split': split_name}))
raw_df = pd.concat(frames, ignore_index=True)
print(f'Pooled: {len(raw_df)} rows.  Labels: {raw_df["label"].value_counts().to_dict()}')

### 1.1 Class Distribution

In [ ]:
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for Colab
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.size'] = 11

fig, ax = plt.subplots(figsize=(5, 3))
raw_df['label'].value_counts().plot.bar(ax=ax, color=['#2ca02c','#1f77b4','#d62728'])
ax.set_title('Sentiment class distribution (raw)')
ax.set_ylabel('count')
plt.tight_layout()
plt.show()

### 1.2 Text Length Distribution

In [ ]:
raw_df['char_len'] = raw_df['text'].str.len()
fig, ax = plt.subplots(figsize=(6, 3))
raw_df['char_len'].hist(bins=50, ax=ax, alpha=0.7)
ax.set_xlabel('Character length')
ax.set_title('Text length distribution (raw)')
plt.tight_layout()
plt.show()
print(raw_df['char_len'].describe())

### 1.3 Sample Texts

In [ ]:
for label in scfg['labels']:
    print(f'\n=== {label.upper()} ===')
    for t in raw_df[raw_df['label'] == label]['text'].sample(3, random_state=SEED).values:
        print(f'  {t[:120]}')

## 2. Preprocessing Pipeline

In [ ]:
from src.preprocessing.text_cleaner import CleaningReport, clean_frame, verify_no_leakage

report = CleaningReport()
report.log('raw_pooled', len(raw_df), len(raw_df))
clean_df = clean_frame(raw_df, ['text'], {'text': scfg['min_chars']}, report,
                       drop_conflicting_labels='label')
clean_df.insert(0, 'id', [f'sent-{i:05d}' for i in range(len(clean_df))])
print(report.to_frame().to_string(index=False))
print(f'\nCleaned: {len(clean_df)} rows')

## 3. Stratified Train/Validation Split

In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(clean_df, test_size=scfg['val_ratio'],
                                    stratify=clean_df['label'], random_state=SEED)
train_df = train_df.sort_values('id').reset_index(drop=True)
val_df = val_df.sort_values('id').reset_index(drop=True)
print(f'Train: {len(train_df)}   Val: {len(val_df)}')
print('\nTrain labels:', train_df['label'].value_counts().to_dict())
print('Val labels:  ', val_df['label'].value_counts().to_dict())

In [ ]:
# Verify zero leakage
overlaps = verify_no_leakage({'train': train_df, 'val': val_df}, 'text')
print('Cross-split overlaps:', overlaps)

## 4. Summarization Dataset

In [ ]:
zcfg = cfg['datasets']['summarization']
ds_sum = None
for cand in zcfg['hf_candidates']:
    try:
        ds_sum = load_dataset(cand['path'], cand.get('name'), trust_remote_code=True)
        print(f'Loaded: {cand}')
        break
    except Exception as e:
        print(f'  skip {cand}: {e}')
print(ds_sum)

In [ ]:
test_df = pd.DataFrame({
    'source': ds_sum['test'][zcfg['source_column']],
    'target': ds_sum['test'][zcfg['target_column']]
})
test_df['src_len'] = test_df['source'].str.len()
test_df['tgt_len'] = test_df['target'].str.len()

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
test_df['src_len'].hist(bins=40, ax=axes[0], alpha=0.7)
axes[0].set_title('Article length (chars)')
test_df['tgt_len'].hist(bins=40, ax=axes[1], alpha=0.7, color='orange')
axes[1].set_title('Summary length (chars)')
plt.tight_layout()
plt.show()
print(f'Compression ratio (median): {(test_df["tgt_len"] / test_df["src_len"]).median():.3f}')

## 5. Save Preprocessed Data

Run the full data-preparation pipeline (same logic as `scripts/prepare_data.py`):

In [ ]:
from src.preprocessing.data_prep import prepare_all

manifest = prepare_all(cfg, SEED)
print(json.dumps({k: manifest[k] for k in ('sentiment', 'summarization')},
                 ensure_ascii=False, indent=2, default=str)[:3000])

## 6. Summary

| Dataset | Task | Cleaned size | Train | Val / Eval |
|---|---|---|---|---|
| AfriSenti amh | Sentiment (3-class) | see above | 80 % | 20 % |
| XL-Sum amharic | Summarization | see above | train+val splits | 500 (eval) |

All processed files are in `data/processed/`. Run `python scripts/generate_results.py`
after training to build tables and figures.